# Задание 3. Аномалии и отказы оборудования

In [ ]:
import os
import warnings

import boto3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from botocore.client import Config
from sqlalchemy import create_engine, text

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

PG_HOST = os.getenv("POSTGRES_HOST", "localhost")
PG_URL = (
    f"postgresql+psycopg2://{os.getenv('POSTGRES_USER', 'oiluser')}:"
    f"{os.getenv('POSTGRES_PASSWORD', 'oilpass')}@"
    f"{PG_HOST}:{os.getenv('POSTGRES_PORT', '5432')}/"
    f"{os.getenv('POSTGRES_DB', 'oildb')}"
)
engine = create_engine(PG_URL)

def minio_endpoint_url():
    raw = os.getenv("MINIO_ENDPOINT", "localhost:9000").strip().rstrip("/")
    if raw.startswith("http://") or raw.startswith("https://"):
        return raw
    return f"http://{raw}"

def minio_client():
    return boto3.client(
        "s3",
        endpoint_url=minio_endpoint_url(),
        aws_access_key_id=os.getenv("MINIO_ACCESS_KEY", "minioadmin"),
        aws_secret_access_key=os.getenv("MINIO_SECRET_KEY", "minioadmin"),
        config=Config(signature_version="s3v4"),
        region_name="us-east-1",
    )

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler



In [ ]:

sensors = pd.read_sql("SELECT * FROM pump_sensors", engine)
failures = pd.read_sql("SELECT * FROM pump_failures", engine)
sensors["timestamp"] = pd.to_datetime(sensors["timestamp"])
failures["failure_date"] = pd.to_datetime(failures["failure_date"])

cols = ["temperature", "vibration", "current", "rpm", "pressure"]
z = np.abs((sensors[cols] - sensors[cols].mean()) / sensors[cols].std())
sensors["zscore_anomaly"] = (z.max(axis=1) > 2.5).astype(int)

iso = IsolationForest(contamination=0.1, random_state=42)
sensors["iso_anomaly"] = (iso.fit_predict(StandardScaler().fit_transform(sensors[cols])) == -1).astype(int)
sensors["is_anomaly"] = ((sensors["zscore_anomaly"] + sensors["iso_anomaly"]) >= 1).astype(int)

risk = sensors.groupby("pump_id").agg(
    anomaly_rate=("is_anomaly", "mean"),
    max_vibration=("vibration", "max"),
).reset_index()
risk["risk_score"] = (0.6*risk["anomaly_rate"] + 0.4*risk["max_vibration"]/risk["max_vibration"].max()).round(3)
print(risk.sort_values("risk_score", ascending=False))

for _, f in failures.iterrows():
    w = sensors[(sensors.pump_id==f.pump_id) & (sensors.timestamp >= f.failure_date - pd.Timedelta(hours=24)) & (sensors.timestamp < f.failure_date)]
    if len(w):
        print(f"Pump {f.pump_id} before {f.failure_type}: vib max={w.vibration.max():.1f}, temp mean={w.temperature.mean():.1f}")



In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sensors.groupby(sensors.timestamp.dt.date)["is_anomaly"].sum().plot(ax=axes[0], marker="o")
axes[0].set_title("Аномалии по времени")

for pid in sensors.pump_id.unique():
    sub = sensors[sensors.pump_id==pid]
    axes[1].plot(sub.timestamp, sub.vibration, label=f"pump {pid}")
axes[1].set_title("Вибрация перед отказом")
axes[1].legend(fontsize=8)

risk.plot.bar(x="pump_id", y="risk_score", ax=axes[2], legend=False)
axes[2].set_title("Risk score по насосам")
plt.tight_layout()
plt.show()

